# Taxonomy Hierarchical Classification - Zero-Shot with Numpy (Enhanced)

In [1]:
# 1) Setup and imports

import numpy as np
import pandas as pd
from collections import defaultdict
from typing import Dict, List, Set, Tuple, Optional
from pathlib import Path
from sentence_transformers import SentenceTransformer

# Load Kedro context
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project

# Bootstrap the project (run from notebook directory)
project_path = Path.cwd().parent
bootstrap_project(project_path)

# Create session and get context
session = KedroSession.create(project_path=project_path)
context = session.load_context()

# Access catalog and parameters
catalog = context.catalog
params = context.params

print("Kedro context loaded successfully")
print(f"Model: {params['model_name']}")
print(f"Taxonomy: {params['taxonomy_key']}")


[01/12/26 17:30:45] INFO     Using                                                                  ]8;id=48767;file:///Users/gabriele/anaconda3/envs/taxomind-env/lib/python3.13/site-packages/kedro/framework/project/__init__.py\__init__.py]8;;\:]8;id=59786;file:///Users/gabriele/anaconda3/envs/taxomind-env/lib/python3.13/site-packages/kedro/framework/project/__init__.py#269\269]8;;\
                             '/Users/gabriele/anaconda3/envs/taxomind-env/lib/python3.13/site-packa                
                             ges/kedro/framework/project/rich_logging.yml' as logging                              
                             configuration.                                                                        

Kedro context loaded successfully
Model: nomic-ai/nomic-embed-text-v2-moe
Taxonomy: ISCO


In [2]:
# 2) Load taxonomy from Kedro catalog

taxonomy_dict = catalog.load("taxonomy_definition")
taxonomy_key = params["taxonomy_key"]
taxonomy = taxonomy_dict.get(taxonomy_key)()

if taxonomy is None:
    raise ValueError(f"Taxonomy '{taxonomy_key}' not found. Available: {list(taxonomy_dict.keys())}")

print(f"Loaded taxonomy: {taxonomy_key}")
print(f"Shape: {taxonomy.shape}")

# Validate and normalize
required = ["level","code","label","definition","examples","parentCode","isLeaf"]
missing = [c for c in required if c not in taxonomy.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

taxonomy = taxonomy.copy()
taxonomy["code"] = taxonomy["code"].astype(str)
taxonomy["parentCode"] = taxonomy["parentCode"].astype(str)
taxonomy["level"] = taxonomy["level"].astype(int)

if taxonomy["isLeaf"].dtype != bool:
    taxonomy["isLeaf"] = taxonomy["isLeaf"].astype(str).str.lower().isin(["true","1","yes","y"])

print(f"\nTaxonomy statistics:")
print(f"  Total nodes: {len(taxonomy)}")
print(f"  Levels: {taxonomy['level'].min()} - {taxonomy['level'].max()}")
print(f"  Leaf nodes: {taxonomy['isLeaf'].sum()}")

taxonomy.head(5)

[01/12/26 17:30:46] INFO     Loading data from taxonomy_definition (PartitionedDataset)...     ]8;id=225336;file:///Users/gabriele/anaconda3/envs/taxomind-env/lib/python3.13/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=588322;file:///Users/gabriele/anaconda3/envs/taxomind-env/lib/python3.13/site-packages/kedro/io/data_catalog.py#1048\1048]8;;\

Loaded taxonomy: ISCO
Shape: (619, 8)

Taxonomy statistics:
  Total nodes: 619
  Levels: 1 - 4
  Leaf nodes: 436


,level,code,label,definition,examples,id,parentCode,isLeaf
0,1,1,Managers,"Managers plan, direct, coordinate and evaluate...",managers usually include formulating and advis...,fe6187ee,nan,False
1,2,11,"Chief Executives, Senior Officials and Legisla...","Chief executives, senior officials and legisla...",workers in this submajor group usually include...,eebe7f94,1,False
2,3,111,Legislators and Senior Officials,"Legislators and senior officials determine, fo...",presiding over or participating in the proceed...,3891a5dd,11,False
3,4,1111,Legislators,"Legislators determine, formulate, and direct p...",(a) presiding over or participating in the pr...,6a5bf9e5,111,True
4,4,1112,Senior Government Officials,Senior government officials advise governments...,"(a) advising national, state, regional or loc...",ed2a37c5,111,True


In [3]:
# 3) Build taxonomy adjacency (parent-child relationships)

code_to_row: Dict[str, dict] = taxonomy.set_index("code").to_dict(orient="index")

children: Dict[str, List[str]] = {code: [] for code in taxonomy["code"]}
parent: Dict[str, str] = {}

for _, row in taxonomy.iterrows():
    code = str(row["code"])
    pcode = str(row["parentCode"]) if pd.notna(row["parentCode"]) else ""
    if pcode and pcode in code_to_row:
        parent[code] = pcode
        children[pcode].append(code)

roots = [code for code in taxonomy["code"] if code not in parent]
print(f"Roots (forest size): {len(roots)}")
print("Example roots:", roots[:10])

# Helper functions
def is_ancestor_or_equal(a: str, b: str, parent) -> bool:
    """Check if a is ancestor of b (or a == b)."""
    cur = b
    if a == b:
        return True
    while cur in parent:
        cur = parent[cur]
        if cur == a:
            return True
    return False

def ancestors_of(nodes: Set[str]) -> Set[str]:
    """Get all ancestors of a set of nodes."""
    out: Set[str] = set()
    for n in nodes:
        cur = n
        while cur in parent:
            cur = parent[cur]
            if cur in out:
                break
            out.add(cur)
    return out

def children_of(nodes: Set[str]) -> Set[str]:
    """Get all children of a set of nodes."""
    out: Set[str] = set()
    for n in nodes:
        out.update(children.get(n, []))
    return out

def roots_of(nodes: Set[str]) -> Set[str]:
    """Find root nodes for a set of candidates."""
    out: Set[str] = set()
    for n in nodes:
        cur = n
        while cur in parent:
            cur = parent[cur]
        out.add(cur)
    return out

print(f"\n✅ Taxonomy graph built")

Roots (forest size): 10
Example roots: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '0']

✅ Taxonomy graph built


In [4]:
# 4) Build multi-view embedding index with sentence-transformers + numpy
#
# Creates separate embeddings for:
# - E_label: Always present (used for retrieval)
# - E_definition: Only if definition exists (used for re-ranking)
# - E_examples: Only if examples exist (used for re-ranking)

MODEL = params["model_name"]
zero_shot_config = params.get("zero_shot", {})

print(f"Loading embedding model: {MODEL}")
model = SentenceTransformer(MODEL, trust_remote_code=True)

# Get embedding dimension
test_embed = model.encode(["test"])
embedding_dim = test_embed.shape[1]
print(f"Embedding dimension: {embedding_dim}")

# Create code-to-index mapping (for fast lookup)
all_codes = taxonomy["code"].tolist()
code_to_idx = {code: idx for idx, code in enumerate(all_codes)}
idx_to_code = {idx: code for code, idx in code_to_idx.items()}

# === Label embeddings (always present, used for retrieval) ===
all_labels = [str(code_to_row[code]["label"]) for code in all_codes]

print(f"\nEmbedding {len(all_labels)} taxonomy labels...")
label_embeddings = model.encode(
    all_labels,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True  # L2 normalization for cosine similarity
)

print(f"Label embeddings shape: {label_embeddings.shape}")

# === Definition embeddings (only where present) ===
def_mask = taxonomy["definition"].notna()
def_count = def_mask.sum()

if def_count > 0:
    print(f"\nEmbedding {def_count} taxonomy definitions...")
    def_texts = taxonomy.loc[def_mask, "definition"].fillna("").tolist()
    def_embeddings_temp = model.encode(
        def_texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    # Create sparse storage (only for nodes with definitions)
    def_codes_with_def = taxonomy.loc[def_mask, "code"].tolist()
    def_code_to_idx = {code: i for i, code in enumerate(def_codes_with_def)}
    def_embeddings = def_embeddings_temp
    print(f"Definition embeddings shape: {def_embeddings.shape}")
else:
    def_code_to_idx = {}
    def_embeddings = None
    print("\nNo definitions found, skipping definition embeddings")

# === Example embeddings (only where present) ===
ex_mask = taxonomy["examples"].notna()
ex_count = ex_mask.sum()

if ex_count > 0:
    print(f"\nEmbedding {ex_count} taxonomy examples...")
    ex_texts = taxonomy.loc[ex_mask, "examples"].fillna("").tolist()
    ex_embeddings_temp = model.encode(
        ex_texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    # Create sparse storage (only for nodes with examples)
    ex_codes_with_ex = taxonomy.loc[ex_mask, "code"].tolist()
    ex_code_to_idx = {code: i for i, code in enumerate(ex_codes_with_ex)}
    ex_embeddings = ex_embeddings_temp
    print(f"Example embeddings shape: {ex_embeddings.shape}")
else:
    ex_code_to_idx = {}
    ex_embeddings = None
    print("\nNo examples found, skipping example embeddings")

print(f"\n✅ Multi-view embedding index built")
print(f"   - {len(all_codes)} nodes")
print(f"   - {embedding_dim} dimensions")
print(f"   - Label embeddings: {len(all_codes)} (100%)")
print(f"   - Definition embeddings: {def_count} ({def_count/len(all_codes)*100:.1f}%)")
print(f"   - Example embeddings: {ex_count} ({ex_count/len(all_codes)*100:.1f}%)")
print(f"   - Normalized for cosine similarity")

Loading embedding model: nomic-ai/nomic-embed-text-v2-moe


[01/12/26 17:30:51] WARNING  /Users/gabriele/.cache/huggingface/modules/transformers_modules/nomic_ ]8;id=770636;file:///Users/gabriele/anaconda3/envs/taxomind-env/lib/python3.13/warnings.py\warnings.py]8;;\:]8;id=965538;file:///Users/gabriele/anaconda3/envs/taxomind-env/lib/python3.13/warnings.py#110\110]8;;\
                             hyphen_ai/nomic_hyphen_bert_hyphen_2048/7710840340a098cfb869c4f65e87cf                
                             2b1b70caca/modeling_hf_nomic_bert.py:1634: UserWarning: Install                       
                             Nomic's megablocks fork for better speed: `pip install                                
                             git+https://github.com/nomic-ai/megablocks.git`                                       
                               warnings.warn("Install Nomic's megablocks fork for better speed: " +                
                                                                                                                   

Embedding dimension: 768

Embedding 619 taxonomy labels...


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Label embeddings shape: (619, 768)

Embedding 619 taxonomy definitions...


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Definition embeddings shape: (619, 768)

Embedding 619 taxonomy examples...


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Example embeddings shape: (619, 768)

✅ Multi-view embedding index built
   - 619 nodes
   - 768 dimensions
   - Label embeddings: 619 (100%)
   - Definition embeddings: 619 (100.0%)
   - Example embeddings: 619 (100.0%)
   - Normalized for cosine similarity


In [5]:
def ancestors_including_self(code: str, parent: dict) -> list[str]:
    """
    Returns [code, parent(code), parent(parent(code)), ...] up to root.
    Root is identified by missing/None parent.
    """
    out = []
    cur = code
    seen = set()
    while cur is not None and cur not in seen:
        out.append(cur)
        seen.add(cur)
        cur = parent.get(cur)
    return out


def path_to_root(code: str, parent: dict) -> list[str]:
    """
    Returns [root, ..., code].
    """
    anc = ancestors_including_self(code, parent)
    return list(reversed(anc))


def ancestor_at_level(code: str, target_level: int, parent: dict, level: dict) -> str | None:
    """
    Walk up until node is at target_level. Returns None if cannot reach.
    """
    cur = code
    seen = set()
    while cur is not None and cur not in seen:
        seen.add(cur)
        if level.get(cur) == target_level:
            return cur
        # if we went above target level (smaller number), stop
        if level.get(cur, 10**9) < target_level:
            return None
        cur = parent.get(cur)
    return None

In [6]:
# 5) Global Retrieval + Candidate Set Construction (Top-K at any level)

K_RETRIEVAL = int(params.get("top_k", 20))  # default 20
M_TOP = int(params.get("top_m", 15))        # top-M for stopping decision
STOP_MODE = params.get("stop_mode", "topm") # "topm" or "lca"

print(f"Retrieval config: K={K_RETRIEVAL}, M={M_TOP}, STOP_MODE={STOP_MODE}")

# Embed query
query = params.get("query_text", "education")
qvec = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]

# ANN retrieval over label embeddings
# label_embeddings: numpy array aligned with taxonomy['code'] in code_to_idx
sims_all = label_embeddings @ qvec  # dot == cosine because normalized
top_idx = np.argsort(-sims_all)[:K_RETRIEVAL]
retrieved = [(idx_to_code[i], float(sims_all[i])) for i in top_idx]

print("\n" + "="*80)
print(f"RETRIEVAL FOR QUERY: '{query}'")
print("="*80)
for rank,(code,sim) in enumerate(retrieved[:20], start=1):
    row = code_to_row[code]
    print(f"{rank:>3}. {code:6s} | L{row['level']} | {row['label'][:45]:45s} → sim={sim:.3f}")

retrieved_codes = [c for c,_ in retrieved]
retrieved_set = set(retrieved_codes)

# Candidate nodes = retrieved + all their ancestors
cand_nodes = set()
for c in retrieved_codes:
    cand_nodes.update(ancestors_including_self(c, parent))

print(f"\nCandidate nodes (retrieved + ancestors): {len(cand_nodes)}")

Retrieval config: K=20, M=15, STOP_MODE=topm

RETRIEVAL FOR QUERY: 'education'
  1. 1345   | L4 | Education Managers                            → sim=0.548
  2. 232    | L3 | Vocational education teachers                 → sim=0.544
  3. 231    | L3 | University and Higher Education Teachers      → sim=0.512
  4. 2310   | L4 | University and Higher Education Teachers      → sim=0.512
  5. 2330   | L4 | Secondary Education Teachers                  → sim=0.507
  6. 233    | L3 | Secondary Education Teachers                  → sim=0.507
  7. 2320   | L4 | Vocational Education Teachers                 → sim=0.480
  8. 2342   | L4 | Early Childhood Educators                     → sim=0.467
  9. 2355   | L4 | Other Arts Teachers                           → sim=0.448
 10. 2341   | L4 | Primary School Teachers                       → sim=0.444
 11. 2351   | L4 | Education Methods Specialists                 → sim=0.434
 12. 234    | L3 | Primary School and Early Childhood Teachers   → sim=0.4

In [7]:
# 6) Multi-View Re-Ranking
#
# After retrieving top-K candidates using label embeddings, re-rank them using
# multi-view scoring to boost precision:
#
# score(n) = max(
#     qvec @ label[n],
#     qvec @ definition[n]   if definition exists else -inf,
#     qvec @ examples[n]     if examples exist else -inf
# )
#
# This allows nodes with strong semantic matches in definitions or examples
# to rank higher, even if their labels are less similar.

def compute_multiview_score(code: str, qvec: np.ndarray) -> Tuple[float, str]:
    """
    Compute multi-view max score for a given code.
    
    Returns:
        (max_score, best_view) where best_view is one of: 'label', 'definition', 'examples'
    """
    # Label score (always present)
    label_sim = float(label_embeddings[code_to_idx[code]] @ qvec)
    
    # Definition score (if exists)
    if code in def_code_to_idx:
        def_sim = float(def_embeddings[def_code_to_idx[code]] @ qvec)
    else:
        def_sim = -np.inf
    
    # Example score (if exists)
    if code in ex_code_to_idx:
        ex_sim = float(ex_embeddings[ex_code_to_idx[code]] @ qvec)
    else:
        ex_sim = -np.inf
    
    # Return max score and which view achieved it
    scores = [
        (label_sim, 'label'),
        (def_sim, 'definition'),
        (ex_sim, 'examples')
    ]
    max_score, best_view = max(scores, key=lambda x: x[0])
    
    return max_score, best_view

# Re-rank retrieved candidates using multi-view scoring
retrieved_reranked = []
for code, original_sim in retrieved:
    multiview_score, best_view = compute_multiview_score(code, qvec)
    retrieved_reranked.append((code, multiview_score, original_sim, best_view))

# Sort by multi-view score (descending)
retrieved_reranked.sort(key=lambda x: x[1], reverse=True)

# Update retrieved list with re-ranked results (keeping original format for compatibility)
retrieved = [(code, mv_score) for code, mv_score, _, _ in retrieved_reranked]
retrieved_codes = [c for c, _ in retrieved]

print("\n" + "="*80)
print("MULTI-VIEW RE-RANKING RESULTS")
print("="*80)
print(f"Top-{min(20, len(retrieved_reranked))} after re-ranking:\n")
for rank, (code, mv_score, orig_sim, best_view) in enumerate(retrieved_reranked[:20], start=1):
    row = code_to_row[code]
    boost = mv_score - orig_sim
    boost_marker = f"↑{boost:+.3f}" if boost > 0.01 else ""
    view_marker = f"[{best_view[0].upper()}]" if best_view != 'label' else ""
    print(f"{rank:>3}. {code:6s} | L{row['level']} | {row['label'][:40]:40s} → {mv_score:.3f} {view_marker:3s} {boost_marker}")

print(f"\nRe-ranking statistics:")
label_best = sum(1 for _, _, _, view in retrieved_reranked if view == 'label')
def_best = sum(1 for _, _, _, view in retrieved_reranked if view == 'definition')
ex_best = sum(1 for _, _, _, view in retrieved_reranked if view == 'examples')
print(f"  Best view = label:      {label_best}/{len(retrieved_reranked)} ({label_best/len(retrieved_reranked)*100:.1f}%)")
print(f"  Best view = definition: {def_best}/{len(retrieved_reranked)} ({def_best/len(retrieved_reranked)*100:.1f}%)")
print(f"  Best view = examples:   {ex_best}/{len(retrieved_reranked)} ({ex_best/len(retrieved_reranked)*100:.1f}%)")

# # Rebuild candidate nodes with re-ranked retrieved set
# cand_nodes = set()
# for c in retrieved_codes:
#     cand_nodes.update(ancestors_including_self(c, parent))



MULTI-VIEW RE-RANKING RESULTS
Top-20 after re-ranking:

  1. 1345   | L4 | Education Managers                       → 0.548     
  2. 232    | L3 | Vocational education teachers            → 0.544     
  3. 231    | L3 | University and Higher Education Teachers → 0.512     
  4. 2310   | L4 | University and Higher Education Teachers → 0.512     
  5. 2330   | L4 | Secondary Education Teachers             → 0.507     
  6. 233    | L3 | Secondary Education Teachers             → 0.507     
  7. 2320   | L4 | Vocational Education Teachers            → 0.480     
  8. 2342   | L4 | Early Childhood Educators                → 0.467     
  9. 2355   | L4 | Other Arts Teachers                      → 0.448     
 10. 2341   | L4 | Primary School Teachers                  → 0.444     
 11. 2351   | L4 | Education Methods Specialists            → 0.434     
 12. 234    | L3 | Primary School and Early Childhood Teach → 0.434     
 13. 2354   | L4 | Other Music Teachers                     → 0.427

In [8]:
# 7) Root-level clustering → Beam roots + Induced subgraph V (root-scoped)
#
# Goal:
# - Use retrieval evidence to pick a small set of plausible L1 roots (beam)
# - Build an induced subgraph V that contains:
#   - retrieved nodes under those roots
#   - all ancestors (to connect paths)
#   - siblings (children of ancestors) so margin/dispersion stopping is well-defined
#
# Inputs expected to already exist in the notebook:
# - retrieved: List[Tuple[str, float]]   # (code, mv_score) after reranking (multi-view score)
# - parent: dict[str, str|None]          # child -> parent
# - children: dict[str, list[str]]       # parent -> children
# - code_to_row: dict[str, dict]         # metadata incl. level,label,isLeaf
#
# Outputs:
# - beam_roots: list[str]
# - V: set[str]                          # induced node set for routing/stopping/validation
# - root_clusters: dict[root -> list[(code,score)]]

from collections import defaultdict

BEAM_K = int(params.get("beam_k", 3))
ROOT_TOP_N = int(params.get("root_top_n", 5))          # aggregate using top-N members per root
COUNT_BONUS_W = float(params.get("root_count_bonus", 0.02))
COUNT_BONUS_CAP = int(params.get("root_count_cap", 10))

def get_root_ancestor(code: str, parent: dict) -> str:
    """Walk up to the root (node with no parent). Robust to cycles."""
    cur = code
    seen = set()
    while cur is not None and cur not in seen:
        seen.add(cur)
        p = parent.get(cur)
        if p is None:
            return cur
        cur = p
    return cur  # fallback if cycle


# ---- 1) Cluster retrieved nodes by root
root_clusters = defaultdict(list)
for code, mv_score in retrieved:
    root = get_root_ancestor(code, parent)
    root_clusters[root].append((code, float(mv_score)))

# ---- 2) Score roots by mean(top-N) + small capped count bonus
root_scores = {}
for root, members in root_clusters.items():
    scores = sorted([s for _, s in members], reverse=True)
    top_scores = scores[:max(1, min(ROOT_TOP_N, len(scores)))]
    mean_top = sum(top_scores) / len(top_scores)

    count_bonus = COUNT_BONUS_W * min(len(members), COUNT_BONUS_CAP)
    root_scores[root] = mean_top + count_bonus

root_ranked = sorted(root_scores.items(), key=lambda x: x[1], reverse=True)
beam_roots = [r for r, _ in root_ranked[:max(1, BEAM_K)]]

print("\n" + "="*80)
print(f"ROOT-LEVEL CLUSTERING → BEAM ROOTS (K={BEAM_K})")
print("="*80)
for i, (root, rs) in enumerate(root_ranked[:max(10, BEAM_K)], 1):
    row = code_to_row.get(root, {"label": "<?>", "level": "?"})
    members = sorted(root_clusters[root], key=lambda x: x[1], reverse=True)
    print(f"{i:>2}. {root:6s} | L{row['level']} | {row['label'][:45]:45s} → root_score={rs:.3f} (hits={len(members)})")
    for code, mv in members[:5]:
        r = code_to_row.get(code, {"label": "<?>", "level": "?"})
        print(f"     └─ {code:6s} | L{r['level']} | {r['label'][:40]:40s} (mv={mv:.3f})")
    if i == BEAM_K:
        print("     ↑ selected for beam")
print(f"\nBeam roots: {beam_roots}")

# ---- 3) Build induced subgraph V restricted to beam roots
# Start from retrieved nodes that fall under selected roots
retrieved_under_beam = [(c, s) for (c, s) in retrieved if any(is_ancestor_or_equal(r, c, parent) for r in beam_roots)]
retrieved_codes_under_beam = [c for (c, _) in retrieved_under_beam]

V = set()

# (a) Add beam roots
V.update(beam_roots)

# (b) Add retrieved nodes under beam + their ancestors up to root
for code in retrieved_codes_under_beam:
    for a in ancestors_including_self(code, parent):
        # Keep only nodes still under one of the beam roots
        if any(is_ancestor_or_equal(r, a, parent) for r in beam_roots):
            V.add(a)

# (c) Add sibling sets (children of every ancestor in V), within the same beam root
# This is critical so margin/dispersion stopping has the full sibling set.
to_expand = list(V)
for p in to_expand:
    for ch in children.get(p, []):
        if any(is_ancestor_or_equal(r, ch, parent) for r in beam_roots):
            V.add(ch)

print("\n" + "="*80)
print("INDUCED SUBGRAPH V (root-scoped)")
print("="*80)
print(f"Retrieved hits total: {len(retrieved)}")
print(f"Retrieved hits under beam roots: {len(retrieved_under_beam)}")
print(f"Induced V size: {len(V)}")

# Convenience outputs some later cells may expect
retrieved_codes = set(retrieved_codes_under_beam)


ROOT-LEVEL CLUSTERING → BEAM ROOTS (K=3)
 1. 2      | L1 | Professionals                                 → root_score=0.717 (hits=17)
     └─ 232    | L3 | Vocational education teachers            (mv=0.544)
     └─ 231    | L3 | University and Higher Education Teachers (mv=0.512)
     └─ 2310   | L4 | University and Higher Education Teachers (mv=0.512)
     └─ 2330   | L4 | Secondary Education Teachers             (mv=0.507)
     └─ 233    | L3 | Secondary Education Teachers             (mv=0.507)
 2. 1      | L1 | Managers                                      → root_score=0.568 (hits=1)
     └─ 1345   | L4 | Education Managers                       (mv=0.548)
 3. 9      | L1 | Elementary Occupations                        → root_score=0.408 (hits=1)
     └─ 9      | L1 | Elementary Occupations                   (mv=0.388)
     ↑ selected for beam
 4. 5      | L1 | Service and Sales Workers                     → root_score=0.395 (hits=1)
     └─ 5312   | L4 | Teachers' Aides         

In [11]:
# 7.5) Ancestor Support + Multi-View Scores (scoped to beam roots)
#
# Compute:
# - support[node]: weighted votes from retrieved descendants (normalized)
# - sim_multiview[node]: multi-view similarity scores for all nodes in V
#
# Key: Support is ONLY computed within the selected beam subtrees,
# preventing semantic dilution across unrelated taxonomy branches.

from collections import defaultdict

GAMMA = float(params.get("gamma", 0.75))

# ===== SUPPORT COMPUTATION (scoped to beam roots) =====
# Only retrieved nodes under beam roots contribute to support
support_raw = defaultdict(float)
total_w = 0.0

for code, sim in retrieved_under_beam:  # Only beam-scoped retrieved nodes
    w = max(sim, 0.0)
    total_w += w
    
    # Propagate vote to ancestors, but ONLY within V (beam-scoped)
    for a in ancestors_including_self(code, parent):
        if a in V:  # Critical: only accumulate within induced subgraph
            support_raw[a] += w

# Normalize support to [0, 1]
support = {}
den = total_w if total_w > 0 else 1.0
for a, v in support_raw.items():
    support[a] = v / den

print("\n" + "="*80)
print("ANCESTOR SUPPORT (scoped to beam roots)")
print("="*80)
print(f"Total weight from retrieved: {total_w:.3f}")
print(f"Nodes with support: {len(support)}/{len(V)}")

# Show top support nodes
top_support = sorted(support.items(), key=lambda x: x[1], reverse=True)[:10]
print(f"\nTop-10 nodes by support:")
for code, sup in top_support:
    row = code_to_row[code]
    print(f"  {code:6s} | L{row['level']} | {row['label'][:45]:45s} → support={sup:.3f}")

# ===== MULTI-VIEW SCORES (for all nodes in V) =====
sim_multiview = {}
retrieved_dict = {c: score for c, score in retrieved_under_beam}

for code in V:
    if code in retrieved_dict:
        # Retrieved node: use its multi-view score from Cell 6
        sim_multiview[code] = retrieved_dict[code]
    else:
        # Non-retrieved node (ancestor or sibling): compute on-demand
        # Use label-only score for efficiency (ancestors weren't retrieved for a reason)
        sim_multiview[code] = float(label_embeddings[code_to_idx[code]] @ qvec)

print("\n" + "="*80)
print("MULTI-VIEW SCORES")
print("="*80)
print(f"Nodes with scores: {len(sim_multiview)}/{len(V)}")
print(f"Retrieved nodes (multi-view): {len(retrieved_dict)}")
print(f"Non-retrieved (label-only): {len(sim_multiview) - len(retrieved_dict)}")

# Sanity check
print(f"\n✅ Ready for path scoring")
print(f"   - support: {len(support)} nodes")
print(f"   - sim_multiview: {len(sim_multiview)} nodes")
print(f"   - V: {len(V)} nodes")
print(f"   - gamma: {GAMMA}")



ANCESTOR SUPPORT (scoped to beam roots)
Total weight from retrieved: 8.662
Nodes with support: 25/60

Top-10 nodes by support:
  2      | L1 | Professionals                                 → support=0.892
  23     | L2 | Teaching Professionals                        → support=0.848
  235    | L3 | Other Teaching Professionals                  → support=0.293
  234    | L3 | Primary School and Early Childhood Teachers   → support=0.155
  231    | L3 | University and Higher Education Teachers      → support=0.118
  232    | L3 | Vocational education teachers                 → support=0.118
  233    | L3 | Secondary Education Teachers                  → support=0.117
  1345   | L4 | Education Managers                            → support=0.063
  134    | L3 | Professional Services Managers                → support=0.063
  13     | L2 | Production and Specialized Services Managers  → support=0.063

MULTI-VIEW SCORES
Nodes with scores: 60/60
Retrieved nodes (multi-view): 19
Non-retrieved (

In [12]:
# 8) Two-Stage Depth Decision: Root Selection + Within-Root Stopping
#
# Stage 1: Select best beam root (which taxonomy subtree?)
# Stage 2: Depth decision within selected root (how deep to go?)
#
# Uses support + multi-view scores computed in Cell 7.5
# Path scoring is rooted (starts from selected beam root, not global root)

# ===== STAGE 1: ROOT SELECTION =====
# Score each beam root by aggregating evidence from its retrieved descendants in V

def compute_root_evidence(root: str, V_nodes: set, retrieved_dict: dict) -> dict:
    """
    Compute aggregated evidence for a root from its descendants in V.
    
    Returns dict with:
    - n_retrieved: number of retrieved hits under this root
    - total_mv_score: sum of multi-view scores
    - mean_mv_score: mean of multi-view scores
    - max_mv_score: best single hit score
    - support_mass: sum of support values
    """
    descendants_in_V = [
        c for c in V_nodes 
        if is_ancestor_or_equal(root, c, parent) and c in retrieved_dict
    ]
    
    if not descendants_in_V:
        return {
            "n_retrieved": 0,
            "total_mv_score": 0.0,
            "mean_mv_score": 0.0,
            "max_mv_score": 0.0,
            "support_mass": 0.0,
            "top_hits": []
        }
    
    mv_scores = [retrieved_dict[c] for c in descendants_in_V]
    support_values = [support.get(c, 0.0) for c in descendants_in_V]
    
    return {
        "n_retrieved": len(descendants_in_V),
        "total_mv_score": sum(mv_scores),
        "mean_mv_score": sum(mv_scores) / len(mv_scores),
        "max_mv_score": max(mv_scores),
        "support_mass": sum(support_values),
        "top_hits": sorted(zip(descendants_in_V, mv_scores), key=lambda x: x[1], reverse=True)[:5]
    }

# Compute evidence for each beam root
root_evidence = {}

for root in beam_roots:
    root_evidence[root] = compute_root_evidence(root, V, retrieved_dict)

# Root selection strategy: weighted combination of mean quality + support mass
ROOT_QUALITY_WEIGHT = float(params.get("root_quality_weight", 0.6))
ROOT_SUPPORT_WEIGHT = float(params.get("root_support_weight", 0.4))

def score_root(evidence: dict) -> float:
    """Combine quality (mean MV score) and quantity (support mass)."""
    quality = evidence["mean_mv_score"]
    support_mass = evidence["support_mass"]
    return ROOT_QUALITY_WEIGHT * quality + ROOT_SUPPORT_WEIGHT * support_mass

root_scores_stage1 = {
    root: score_root(evidence) 
    for root, evidence in root_evidence.items()
}

# Select best root
best_root = max(root_scores_stage1.items(), key=lambda x: x[1])[0]
best_root_score = root_scores_stage1[best_root]
best_root_row = code_to_row[best_root]

print("\n" + "="*80)
print("STAGE 1: BEAM ROOT SELECTION")
print("="*80)
for root in beam_roots:
    ev = root_evidence[root]
    sc = root_scores_stage1[root]
    row = code_to_row[root]
    marker = " ← SELECTED" if root == best_root else ""
    print(f"{root:6s} | {row['label'][:45]:45s} → score={sc:.3f} (hits={ev['n_retrieved']}, mean_mv={ev['mean_mv_score']:.3f}, support={ev['support_mass']:.3f}){marker}")
    
    # Show top evidence
    if ev['top_hits']:
        for code, mv in ev['top_hits'][:3]:
            r = code_to_row[code]
            print(f"     └─ {code:6s} | L{r['level']} | {r['label'][:40]:40s} (mv={mv:.3f})")

print(f"\nSelected root: {best_root} | {best_root_row['label']}")

# ===== STAGE 2: DEPTH DECISION WITHIN SELECTED ROOT =====
# Now focus only on nodes under the selected root in V

V_under_root = [c for c in V if is_ancestor_or_equal(best_root, c, parent)]
V_retrieved_under_root = [c for c in V_under_root if c in retrieved_dict]

# Compute path scores only within this subtree
def path_score_rooted(code: str, root: str, gamma: float = GAMMA) -> float:
    """
    Path score from selected root down to code (not from global root).
    
    ScorePath(code) = Σ_{a in path(root→code)} gamma^i * support[a] * sim_multiview[a]
    where i=0 at root (gamma^0=1), i increases toward leaves (discount)
    """
    # Build path from root to code
    path = []
    cur = code
    seen = set()
    while cur is not None and cur not in seen:
        path.append(cur)
        if cur == root:
            break
        seen.add(cur)
        cur = parent.get(cur)
    
    if root not in path:
        # code is not under root, return -inf
        return -1e9
    
    path.reverse()  # now [root, ..., code]
    
    # Compute score with gamma decay (root gets gamma^0=1, leaves get gamma^depth)
    s = 0.0
    for i, a in enumerate(path):
        s += (gamma ** i) * support.get(a, 0.0) * sim_multiview.get(a, 0.0)
    return float(s)

# Rank candidates within selected root by rooted path score
V_ranked = sorted(
    V_under_root, 
    key=lambda c: path_score_rooted(c, best_root), 
    reverse=True
)

# Get top-M candidates within this root for stopping decision
M_WITHIN_ROOT = min(M_TOP, len(V_ranked))
topM_within_root = V_ranked[:M_WITHIN_ROOT]

print("\n" + "="*80)
print(f"STAGE 2: DEPTH DECISION WITHIN ROOT '{best_root}'")
print("="*80)
print(f"Nodes in V under root: {len(V_under_root)}")
print(f"Retrieved nodes under root: {len(V_retrieved_under_root)}")
print(f"\nTop-{M_WITHIN_ROOT} candidates (by rooted path score):")

for i, code in enumerate(topM_within_root[:15], start=1):
    row = code_to_row[code]
    ps = path_score_rooted(code, best_root)
    mv = sim_multiview.get(code, 0.0)
    sup = support.get(code, 0.0)
    leaf_marker = "🍃" if row.get("isLeaf", False) else "  "
    print(f"{i:>2}. {code:6s} | L{row['level']} | {row['label'][:40]:40s} {leaf_marker} → ps={ps:.4f} (mv={mv:.3f}, sup={sup:.3f})")

# ===== STOPPING DECISION =====
# Apply stopping logic to prevent over/under-specification

STOP_MODE_WITHIN = params.get("stop_mode_within_root", "topm")  # or "sibling_margin"

if STOP_MODE_WITHIN == "sibling_margin":
    # Sibling-aware stopping: look for strong consensus among siblings at each level
    SIBLING_MARGIN = float(params.get("sibling_margin", 0.10))
    SIBLING_MIN_SCORE = float(params.get("sibling_min_score", 0.50))
    
    # Start from root and descend level by level
    current = best_root
    trace = [(best_root, path_score_rooted(best_root, best_root), "root")]
    
    for level in range(code_to_row[best_root]["level"] + 1, 10):  # max 10 levels
        # Get children of current node that are in V
        candidates = [c for c in children.get(current, []) if c in V_under_root]
        
        if not candidates:
            # No children in V, stop here
            break
        
        # Score each candidate
        scored = [(c, path_score_rooted(c, best_root)) for c in candidates]
        scored.sort(key=lambda x: x[1], reverse=True)
        
        if len(scored) == 1:
            # Only one child, descend
            current, score = scored[0]
            trace.append((current, score, "only_child"))
            continue
        
        best_child, best_score = scored[0]
        second_child, second_score = scored[1]
        margin = best_score - second_score
        
        # Check stopping conditions
        if margin < SIBLING_MARGIN:
            # Ambiguous siblings, stop at current
            trace.append((current, best_score, f"ambiguous_siblings (margin={margin:.3f})"))
            break
        
        if best_score < SIBLING_MIN_SCORE:
            # Weak evidence, stop at current
            trace.append((current, best_score, f"weak_evidence (score={best_score:.3f})"))
            break
        
        # Strong winner, descend
        current = best_child
        trace.append((current, best_score, f"strong_winner (margin={margin:.3f})"))
    
    final_code = current
    stopping_reason = trace[-1][2]
    
    print("\n" + "="*80)
    print("SIBLING-MARGIN STOPPING TRACE")
    print("="*80)
    for code, score, reason in trace:
        row = code_to_row[code]
        print(f"L{row['level']}: {code:6s} | {row['label'][:50]:50s} → {reason}")

else:
    # Use top-M concentration (mass-based consensus at each level)
    P1_MIN = float(params.get("p1_min", 0.60))
    GAP_MIN = float(params.get("gap_min", 0.10))
    
    def level_mass_rooted(top_nodes: List[str], target_level: int) -> Dict[str, float]:
        """Mass distribution at target_level for nodes in top_nodes."""
        mass = defaultdict(float)
        for n in top_nodes:
            # Find ancestor of n at target_level - inline implementation
            cur = n
            a = None
            seen = set()
            while cur is not None and cur not in seen:
                seen.add(cur)
                if code_to_row[cur]["level"] == target_level:
                    a = cur
                    break
                # Stop if we went above target level
                if code_to_row[cur]["level"] < target_level:
                    break
                cur = parent.get(cur)
            
            if a is not None and is_ancestor_or_equal(best_root, a, parent):
                mass[a] += path_score_rooted(n, best_root)
        
        total = sum(mass.values()) or 1.0
        return {k: v/total for k, v in mass.items()}
    
    # Find max level in topM
    max_level = max(code_to_row[c]["level"] for c in topM_within_root)
    min_level = code_to_row[best_root]["level"]
    
    chosen = None
    trace = []
    
    for L in range(min_level, max_level + 1):
        mass = level_mass_rooted(topM_within_root, L)
        if not mass:
            break
        
        ranked_mass = sorted(mass.items(), key=lambda x: x[1], reverse=True)
        n1, p1 = ranked_mass[0]
        p2 = ranked_mass[1][1] if len(ranked_mass) > 1 else 0.0
        gap = p1 - p2
        trace.append((L, n1, p1, p2, gap))
        
        if p1 >= P1_MIN and gap >= GAP_MIN:
            chosen = n1
            continue
        else:
            # Ambiguity detected, stop at previous chosen
            final_code = chosen if chosen else n1
            stopping_reason = f"ambiguity_at_level_{L}"
            break
    else:
        final_code = chosen if chosen else topM_within_root[0]
        stopping_reason = "max_consensus_depth"
    
    print("\n" + "="*80)
    print("TOP-M CONCENTRATION TRACE")
    print("="*80)
    for L, n1, p1, p2, gap in trace:
        row = code_to_row[n1]
        consensus = "✓" if (p1 >= P1_MIN and gap >= GAP_MIN) else "✗"
        print(f"L{L}: {n1:6s} | {row['label'][:40]:40s} p1={p1:.2f} p2={p2:.2f} gap={gap:.2f} {consensus}")

final_row = code_to_row[final_code]

print("\n" + "="*80)
print("FINAL DECISION (Two-Stage)")
print("="*80)
print(f"Selected root: {best_root} | {best_root_row['label']} (score={best_root_score:.3f})")
print(f"Final node:    {final_code} | L{final_row['level']} | {final_row['label']}")
print(f"Reason: {stopping_reason}")



STAGE 1: BEAM ROOT SELECTION
2      | Professionals                                 → score=1.160 (hits=17, mean_mv=0.455, support=2.218) ← SELECTED
     └─ 232    | L3 | Vocational education teachers            (mv=0.544)
     └─ 2310   | L4 | University and Higher Education Teachers (mv=0.512)
     └─ 231    | L3 | University and Higher Education Teachers (mv=0.512)
1      | Managers                                      → score=0.354 (hits=1, mean_mv=0.548, support=0.063)
     └─ 1345   | L4 | Education Managers                       (mv=0.548)
9      | Elementary Occupations                        → score=0.251 (hits=1, mean_mv=0.388, support=0.045)
     └─ 9      | L1 | Elementary Occupations                   (mv=0.388)

Selected root: 2 | Professionals

STAGE 2: DEPTH DECISION WITHIN ROOT '2'
Nodes in V under root: 37
Retrieved nodes under root: 17

Top-15 candidates (by rooted path score):
 1. 2355   | L4 | Other Arts Teachers                      🍃 → ps=0.5457 (mv=0.448, sup=0

In [13]:
# 9) HiRAG-STYLE SCOPED VALIDATION (beam-aware, rooted scoring)
#
# Validates the final_code from Cell 8 using evidence in topM_within_root
# Uses rooted path scoring (from best_root, not global root)
#
# Logic:
# - If best evidence (L*) is within TD subtree → CONSISTENT
# - If best evidence is outside TD → check if decisive enough to OVERRIDE
# - Otherwise → CONFLICT (need human review)

OVERRIDE_MARGIN = float(params.get("hirag_override_margin", 0.10))
STABILITY_MARGIN = float(params.get("hirag_stability_margin", 0.08))
MASS_IN_MIN = float(params.get("hirag_mass_in_min", 0.55))
MASS_OUT_RATIO = float(params.get("hirag_mass_out_ratio", 1.30))
ALLOW_DEEPER = bool(params.get("hirag_allow_deeper", False))
DEEPER_MARGIN = float(params.get("hirag_deeper_margin", 0.08))

def best_two_rooted(codes: list[str], root: str):
    """Return (best_code, best_score, second_code, second_score) by rooted path_score."""
    if not codes:
        return None, -1e9, None, -1e9
    ranked = sorted(
        ((c, float(path_score_rooted(c, root))) for c in codes), 
        key=lambda x: x[1], 
        reverse=True
    )
    best_c, best_s = ranked[0]
    if len(ranked) > 1:
        sec_c, sec_s = ranked[1]
    else:
        sec_c, sec_s = None, -1e9
    return best_c, best_s, sec_c, sec_s

def pool_mass_rooted(codes: list[str], root: str) -> float:
    """Total evidence mass using rooted path scoring."""
    return float(sum(max(path_score_rooted(c, root), 0.0) for c in codes))

TD = final_code  # From Cell 8

# Validation pool: use topM_within_root (from Cell 8)
pool_codes = list(topM_within_root) if topM_within_root else []
pool_kind = "topM_within_root"

# Prefer leaves if present
pool_leaves = [c for c in pool_codes if bool(code_to_row[c].get("isLeaf", False))]
if pool_leaves:
    pool = pool_leaves
    pool_kind = "topM_within_root_leaf"
else:
    pool = pool_codes

# If pool is empty, fallback
if not pool:
    pool = V_retrieved_under_root if V_retrieved_under_root else list(V)
    pool_kind = "V_retrieved_under_root"

# Find best evidence in pool
L_star, s_star, L2, s2 = best_two_rooted(pool, best_root)

# Best evidence under TD subtree (scoped)
sub_pool = [c for c in pool if is_ancestor_or_equal(TD, c, parent)]
L_sub, s_sub, _, _ = best_two_rooted(sub_pool, best_root)

# Guards
in_subtree = (L_star is not None) and is_ancestor_or_equal(TD, L_star, parent)
stability = s_star - s2 if L2 is not None else 1e9
margin = (s_star - s_sub) if (L_sub is not None) else None

# Mass-based guards
mass_total = pool_mass_rooted(pool, best_root) if pool else 0.0
mass_in = pool_mass_rooted(sub_pool, best_root) if sub_pool else 0.0
mass_out = max(mass_total - mass_in, 0.0)

mass_in_ratio = (mass_in / mass_total) if mass_total > 0 else 0.0
mass_out_ratio = (mass_out / mass_in) if mass_in > 0 else float("inf")

# Decision logic
status = "WEAK_VALIDATION"
decision_scoped = TD
reason = ""

if L_star is None:
    status = "WEAK_VALIDATION"
    reason = "no_pool_evidence"
elif in_subtree:
    # Best evidence is within TD subtree
    if mass_in_ratio >= MASS_IN_MIN:
        status = "CONSISTENT"
        reason = f"best_{pool_kind}_within_TD_and_mass_in_ratio={mass_in_ratio:.2f}"
    else:
        status = "WEAK_VALIDATION"
        reason = f"best_{pool_kind}_within_TD_but_low_mass_in_ratio={mass_in_ratio:.2f}"
else:
    # Best evidence is outside TD subtree
    if L_sub is None:
        status = "CONFLICT"
        reason = f"best_{pool_kind}_outside_TD_and_no_in_subtree_evidence"
    else:
        # Override only if (a) decisive margin+stability AND (b) outside mass dominates
        decisive = (margin is not None) and (margin >= OVERRIDE_MARGIN) and (stability >= STABILITY_MARGIN)
        outside_dominates = (mass_out_ratio >= MASS_OUT_RATIO)

        if decisive and outside_dominates:
            status = "OVERRIDE"
            decision_scoped = L_star
            reason = f"outside_best_decisive (margin={margin:.3f}, stability={stability:.3f}, mass_out/in={mass_out_ratio:.2f})"
        else:
            status = "CONFLICT"
            reason = f"outside_best_not_decisive (margin={margin:.3f}, stability={stability:.3f}, mass_in_ratio={mass_in_ratio:.2f}, mass_out/in={mass_out_ratio:.2f})"

# Optional: allow deepening within subtree when evidence strongly prefers a descendant of TD
if status in ("CONSISTENT", "WEAK_VALIDATION") and ALLOW_DEEPER and L_sub is not None:
    td_score = float(path_score_rooted(TD, best_root))
    if (s_sub - td_score) >= DEEPER_MARGIN and is_ancestor_or_equal(TD, L_sub, parent) and (L_sub != TD):
        status = "DEEPER"
        decision_scoped = L_sub
        reason = f"deepen_within_TD (sub-td margin={s_sub-td_score:.3f})"

print("\n" + "="*80)
print("HiRAG-STYLE SCOPED VALIDATION (beam-aware, rooted scoring)")
print("="*80)
print(f"Selected root: {best_root} | {code_to_row[best_root]['label']}")
print(f"TD (final_code): {TD} | L{code_to_row[TD]['level']} | {code_to_row[TD]['label']}")
print(f"Pool kind: {pool_kind} | pool size={len(pool)}")
print(f"Mass in TD subtree: {mass_in:.4f} / total {mass_total:.4f} => in_ratio={mass_in_ratio:.2f}, out/in={mass_out_ratio:.2f}")
print(f"Decision: {decision_scoped} | Status: {status}")
print(f"Reason: {reason}")

if L_star:
    print(f"\nL* (best in pool): {L_star} | L{code_to_row[L_star]['level']} | {code_to_row[L_star]['label']} | path_score={s_star:.4f}")
if L2:
    print(f"L2 (2nd best in pool): {L2} | path_score={s2:.4f} | stability={stability:.4f}")
if L_sub:
    print(f"L_sub (best under TD): {L_sub} | L{code_to_row[L_sub]['level']} | {code_to_row[L_sub]['label']} | path_score={s_sub:.4f}")
else:
    print("L_sub: None (no pool evidence inside TD subtree)")

# Show validation outcome
print("\n" + "="*80)
print("VALIDATION OUTCOME")
print("="*80)
if status == "CONSISTENT":
    print("✅ CONSISTENT: Top-down decision aligns with bottom-up evidence")
elif status == "OVERRIDE":
    print("⚠️  OVERRIDE: Bottom-up evidence suggests different node")
    print(f"   Suggested: {decision_scoped} | {code_to_row[decision_scoped]['label']}")
elif status == "DEEPER":
    print("⬇️  DEEPER: Evidence suggests more specific node within subtree")
    print(f"   Suggested: {decision_scoped} | {code_to_row[decision_scoped]['label']}")
elif status == "CONFLICT":
    print("❌ CONFLICT: Evidence is ambiguous, human review recommended")
else:
    print("⚠️  WEAK_VALIDATION: Limited evidence, low confidence")



HiRAG-STYLE SCOPED VALIDATION (beam-aware, rooted scoring)
Selected root: 2 | Professionals
TD (final_code): 23 | L2 | Teaching Professionals
Pool kind: topM_within_root_leaf | pool size=12
Mass in TD subtree: 6.3728 / total 6.3728 => in_ratio=1.00, out/in=0.00
Decision: 23 | Status: CONSISTENT
Reason: best_topM_within_root_leaf_within_TD_and_mass_in_ratio=1.00

L* (best in pool): 2355 | L4 | Other Arts Teachers | path_score=0.5457
L2 (2nd best in pool): 2351 | path_score=0.5451 | stability=0.0006
L_sub (best under TD): 2355 | L4 | Other Arts Teachers | path_score=0.5457

VALIDATION OUTCOME
✅ CONSISTENT: Top-down decision aligns with bottom-up evidence


In [14]:
# 10) Validation Against Training Data
#
# Load training data from catalog and validate the classification algorithm
# 
# Metrics computed:
# - Accuracy at each hierarchy level (L1, L2, L3, L4)
# - Hierarchical accuracy (partial credit for ancestor matches)
# - Mean Reciprocal Rank (MRR) of correct answer in top-K
# - Confusion matrix for each level
#
# This helps tune parameters (K, M, gamma, thresholds) and identify failure modes.

import pandas as pd
from typing import List, Tuple
from collections import Counter

print("\n" + "="*80)
print("VALIDATION AGAINST TRAINING DATA")
print("="*80)

# ===== LOAD TRAINING DATA =====
try:
    taxonomy_training = catalog.load("taxonomy_training")
    training_data = taxonomy_training.get(taxonomy_key)
    
    if training_data is None or training_data.empty:
        print(f"⚠️  No training data found for taxonomy '{taxonomy_key}'")
        print("Skipping validation...")
    else:
        print(f"✅ Loaded training data: {len(training_data)} samples")
        print(f"Columns: {list(training_data.columns)}")
        
        # Validate required columns
        required_cols = ["query", "code"]  # Adjust based on your actual column names
        missing_cols = [c for c in required_cols if c not in training_data.columns]
        
        if missing_cols:
            print(f"⚠️  Missing required columns: {missing_cols}")
            print("Available columns:", list(training_data.columns))
            print("Skipping validation...")
        else:
            # ===== HELPER FUNCTION: Run classification for a query =====
            def classify_query(query_text: str, verbose: bool = False) -> dict:
                """
                Run the full classification pipeline for a single query.
                Returns dict with prediction details.
                """
                # Embed query
                qvec_test = model.encode([query_text], convert_to_numpy=True, normalize_embeddings=True)[0]
                
                # Global retrieval
                sims_all_test = label_embeddings @ qvec_test
                top_idx_test = np.argsort(-sims_all_test)[:K_RETRIEVAL]
                retrieved_test = [(idx_to_code[i], float(sims_all_test[i])) for i in top_idx_test]
                
                # Multi-view re-ranking
                retrieved_reranked_test = []
                for code, original_sim in retrieved_test:
                    mv_score, best_view = compute_multiview_score(code, qvec_test)
                    retrieved_reranked_test.append((code, mv_score, original_sim, best_view))
                retrieved_reranked_test.sort(key=lambda x: x[1], reverse=True)
                retrieved_test = [(code, mv_score) for code, mv_score, _, _ in retrieved_reranked_test]
                
                # Root clustering
                root_clusters_test = defaultdict(list)
                for code, mv_score in retrieved_test:
                    root = get_root_ancestor(code, parent)
                    root_clusters_test[root].append((code, float(mv_score)))
                
                # Score roots
                root_scores_test = {}
                for root, members in root_clusters_test.items():
                    scores = sorted([s for _, s in members], reverse=True)
                    top_scores = scores[:max(1, min(ROOT_TOP_N, len(scores)))]
                    mean_top = sum(top_scores) / len(top_scores)
                    count_bonus = COUNT_BONUS_W * min(len(members), COUNT_BONUS_CAP)
                    root_scores_test[root] = mean_top + count_bonus
                
                root_ranked_test = sorted(root_scores_test.items(), key=lambda x: x[1], reverse=True)
                beam_roots_test = [r for r, _ in root_ranked_test[:max(1, BEAM_K)]]
                
                # Build induced subgraph V
                retrieved_under_beam_test = [
                    (c, s) for (c, s) in retrieved_test 
                    if any(is_ancestor_or_equal(r, c, parent) for r in beam_roots_test)
                ]
                
                V_test = set()
                V_test.update(beam_roots_test)
                
                for code, _ in retrieved_under_beam_test:
                    for a in ancestors_including_self(code, parent):
                        if any(is_ancestor_or_equal(r, a, parent) for r in beam_roots_test):
                            V_test.add(a)
                
                to_expand = list(V_test)
                for p in to_expand:
                    for ch in children.get(p, []):
                        if any(is_ancestor_or_equal(r, ch, parent) for r in beam_roots_test):
                            V_test.add(ch)
                
                # Compute support and sim_multiview
                support_test = defaultdict(float)
                total_w_test = 0.0
                
                for code, sim in retrieved_under_beam_test:
                    w = max(sim, 0.0)
                    total_w_test += w
                    for a in ancestors_including_self(code, parent):
                        if a in V_test:
                            support_test[a] += w
                
                den_test = total_w_test if total_w_test > 0 else 1.0
                support_test = {a: v/den_test for a, v in support_test.items()}
                
                sim_multiview_test = {}
                retrieved_dict_test = {c: score for c, score in retrieved_under_beam_test}
                
                for code in V_test:
                    if code in retrieved_dict_test:
                        sim_multiview_test[code] = retrieved_dict_test[code]
                    else:
                        sim_multiview_test[code] = float(label_embeddings[code_to_idx[code]] @ qvec_test)
                
                # Root selection (Stage 1)
                def compute_root_evidence_test(root: str) -> dict:
                    descendants = [c for c in V_test if is_ancestor_or_equal(root, c, parent) and c in retrieved_dict_test]
                    if not descendants:
                        return {"mean_mv_score": 0.0, "support_mass": 0.0}
                    mv_scores = [retrieved_dict_test[c] for c in descendants]
                    support_values = [support_test.get(c, 0.0) for c in descendants]
                    return {
                        "mean_mv_score": sum(mv_scores) / len(mv_scores),
                        "support_mass": sum(support_values)
                    }
                
                root_evidence_test = {root: compute_root_evidence_test(root) for root in beam_roots_test}
                root_scores_stage1_test = {
                    root: ROOT_QUALITY_WEIGHT * ev["mean_mv_score"] + ROOT_SUPPORT_WEIGHT * ev["support_mass"]
                    for root, ev in root_evidence_test.items()
                }
                
                best_root_test = max(root_scores_stage1_test.items(), key=lambda x: x[1])[0]
                
                # Depth decision (Stage 2)
                V_under_root_test = [c for c in V_test if is_ancestor_or_equal(best_root_test, c, parent)]
                
                def path_score_rooted_test(code: str, root: str) -> float:
                    path = []
                    cur = code
                    seen = set()
                    while cur is not None and cur not in seen:
                        path.append(cur)
                        if cur == root:
                            break
                        seen.add(cur)
                        cur = parent.get(cur)
                    if root not in path:
                        return -1e9
                    path.reverse()
                    s = 0.0
                    for i, a in enumerate(path):
                        s += (GAMMA ** i) * support_test.get(a, 0.0) * sim_multiview_test.get(a, 0.0)
                    return float(s)
                
                V_ranked_test = sorted(V_under_root_test, key=lambda c: path_score_rooted_test(c, best_root_test), reverse=True)
                
                if not V_ranked_test:
                    return {
                        "predicted_code": best_root_test,
                        "best_root": best_root_test,
                        "top_k_codes": [best_root_test],
                        "top_k_scores": [0.0],
                        "stopping_reason": "empty_V_under_root"
                    }
                
                # Return top-K predictions for MRR calculation
                top_k_codes = V_ranked_test[:10]
                top_k_scores = [path_score_rooted_test(c, best_root_test) for c in top_k_codes]
                
                return {
                    "predicted_code": V_ranked_test[0],
                    "best_root": best_root_test,
                    "top_k_codes": top_k_codes,
                    "top_k_scores": top_k_scores,
                    "stopping_reason": "path_score_ranking"
                }
            
            # ===== RUN VALIDATION =====
            print("\n" + "="*80)
            print("RUNNING CLASSIFICATION ON TRAINING DATA")
            print("="*80)
            
            results = []
            
            for idx, row in training_data.iterrows():
                query_text = str(row["query"])
                true_code = str(row["code"])
                
                # Skip if true_code not in taxonomy
                if true_code not in code_to_row:
                    continue
                
                # Classify
                pred = classify_query(query_text, verbose=False)
                pred_code = pred["predicted_code"]
                
                # Compute metrics
                exact_match = (pred_code == true_code)
                
                # Hierarchical match (partial credit)
                if exact_match:
                    hierarchical_distance = 0
                elif is_ancestor_or_equal(pred_code, true_code, parent):
                    # Predicted an ancestor (under-specification)
                    hierarchical_distance = code_to_row[true_code]["level"] - code_to_row[pred_code]["level"]
                elif is_ancestor_or_equal(true_code, pred_code, parent):
                    # Predicted a descendant (over-specification)
                    hierarchical_distance = code_to_row[pred_code]["level"] - code_to_row[true_code]["level"]
                else:
                    # Different branch
                    # Find lowest common ancestor
                    true_ancestors = set(ancestors_including_self(true_code, parent))
                    pred_ancestors = set(ancestors_including_self(pred_code, parent))
                    common = true_ancestors & pred_ancestors
                    if common:
                        lca = max(common, key=lambda c: code_to_row[c]["level"])
                        lca_level = code_to_row[lca]["level"]
                        hierarchical_distance = (code_to_row[true_code]["level"] - lca_level) + (code_to_row[pred_code]["level"] - lca_level)
                    else:
                        hierarchical_distance = 10  # No common ancestor
                
                # MRR: find rank of true_code in top_k_codes
                top_k_codes = pred["top_k_codes"]
                if true_code in top_k_codes:
                    rank = top_k_codes.index(true_code) + 1
                    mrr = 1.0 / rank
                else:
                    mrr = 0.0
                
                results.append({
                    "query": query_text,
                    "true_code": true_code,
                    "true_label": code_to_row[true_code]["label"],
                    "true_level": code_to_row[true_code]["level"],
                    "pred_code": pred_code,
                    "pred_label": code_to_row[pred_code]["label"],
                    "pred_level": code_to_row[pred_code]["level"],
                    "best_root": pred["best_root"],
                    "exact_match": exact_match,
                    "hierarchical_distance": hierarchical_distance,
                    "mrr": mrr
                })
            
            results_df = pd.DataFrame(results)
            
            # ===== COMPUTE METRICS =====
            print("\n" + "="*80)
            print("VALIDATION METRICS")
            print("="*80)
            
            # Overall accuracy
            exact_accuracy = results_df["exact_match"].mean()
            print(f"\nExact Match Accuracy: {exact_accuracy:.2%}")
            
            # Hierarchical accuracy (partial credit: 1 / (1 + distance))
            results_df["hierarchical_accuracy"] = 1.0 / (1.0 + results_df["hierarchical_distance"])
            hierarchical_accuracy = results_df["hierarchical_accuracy"].mean()
            print(f"Hierarchical Accuracy: {hierarchical_accuracy:.2%}")
            
            # MRR
            mrr_mean = results_df["mrr"].mean()
            print(f"Mean Reciprocal Rank (MRR@10): {mrr_mean:.3f}")
            
            # Accuracy by level
            print("\n" + "-"*80)
            print("Accuracy by True Label Level:")
            print("-"*80)
            for level in sorted(results_df["true_level"].unique()):
                level_df = results_df[results_df["true_level"] == level]
                level_acc = level_df["exact_match"].mean()
                level_hier_acc = level_df["hierarchical_accuracy"].mean()
                print(f"Level {level}: Exact={level_acc:.2%}, Hierarchical={level_hier_acc:.2%} (n={len(level_df)})")
            
            # Root selection accuracy
            print("\n" + "-"*80)
            print("Root Selection Accuracy:")
            print("-"*80)
            results_df["true_root"] = results_df["true_code"].apply(lambda c: get_root_ancestor(c, parent))
            results_df["root_match"] = results_df["best_root"] == results_df["true_root"]
            root_accuracy = results_df["root_match"].mean()
            print(f"Root matches ground truth: {root_accuracy:.2%}")
            
            # Error analysis
            print("\n" + "-"*80)
            print("Error Analysis:")
            print("-"*80)
            
            errors = results_df[~results_df["exact_match"]]
            print(f"Total errors: {len(errors)}/{len(results_df)} ({len(errors)/len(results_df):.1%})")
            
            if len(errors) > 0:
                # Categorize errors
                over_spec = errors[errors["pred_level"] > errors["true_level"]]
                under_spec = errors[errors["pred_level"] < errors["true_level"]]
                wrong_branch = errors[errors["pred_level"] == errors["true_level"]]
                
                print(f"  - Over-specification (too deep): {len(over_spec)} ({len(over_spec)/len(errors):.1%})")
                print(f"  - Under-specification (too shallow): {len(under_spec)} ({len(under_spec)/len(errors):.1%})")
                print(f"  - Wrong branch (same level): {len(wrong_branch)} ({len(wrong_branch)/len(errors):.1%})")
                
                # Show worst errors (highest hierarchical distance)
                print("\nWorst Errors (top 5 by hierarchical distance):")
                worst = errors.nlargest(5, "hierarchical_distance")
                for idx, row in worst.iterrows():
                    print(f"\nQuery: {row['query'][:60]}")
                    print(f"  True:  L{row['true_level']} {row['true_code']:6s} | {row['true_label'][:40]}")
                    print(f"  Pred:  L{row['pred_level']} {row['pred_code']:6s} | {row['pred_label'][:40]}")
                    print(f"  Distance: {row['hierarchical_distance']}")
            
            # Save results for further analysis
            print("\n" + "="*80)
            print(f"✅ Validation complete: {len(results_df)} samples processed")
            print("="*80)
            
            # Store results in global variable for inspection
            validation_results = results_df

except Exception as e:
    print(f"⚠️  Error loading training data: {e}")
    print("Skipping validation...")
    import traceback
    traceback.print_exc()



VALIDATION AGAINST TRAINING DATA


[01/12/26 17:49:53] INFO     Loading data from taxonomy_training (PartitionedDataset)...       ]8;id=191829;file:///Users/gabriele/anaconda3/envs/taxomind-env/lib/python3.13/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=654748;file:///Users/gabriele/anaconda3/envs/taxomind-env/lib/python3.13/site-packages/kedro/io/data_catalog.py#1048\1048]8;;\

⚠️  Error loading training data: 'function' object has no attribute 'empty'
Skipping validation...


Traceback (most recent call last):
  File "/var/folders/xb/x5q75wc56gqfpn4thn6bgqbh0000gn/T/ipykernel_99821/2950576756.py", line 26, in <module>
    if training_data is None or training_data.empty:
                                ^^^^^^^^^^^^^^^^^^^
AttributeError: 'function' object has no attribute 'empty'
